## Imports

In [ ]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

import neuro_fuzzy_toolbox as nft

In [7]:
import numpy as np

In [8]:
from sklearn.preprocessing import MinMaxScaler

# Surface (1k)

## Data

In [9]:
def z(x, y):
  return ((3) * ((1-x)**2) * (np.exp(-(x**2)-((y+1)**2))) - (10) * ((x/5)-(x**3)-(y**5)) * (np.exp(-(x**2)-(y**2))) - (1/3)*np.exp(-(x+1)**2-(y**2)))

#Training
x0 = np.random.uniform(-3,3,1000)
x1 = np.random.uniform(-3,3,1000)

e = np.random.normal(0,0.5,1000) #noise
Y = z(x0,x1) + e

#Testing
x0_test = np.random.uniform(-3,3,1000)
x1_test = np.random.uniform(-3,3,1000)

Y_test = z(x0_test,x1_test)

In [10]:
#Training
scaler = MinMaxScaler(feature_range=(0, 1))
vstack_train = np.vstack((x0,x1)).T
scaled_train = scaler.fit_transform(vstack_train)

#Testing
vstack_test = np.vstack((x0_test,x1_test)).T
scaled_test = scaler.transform(vstack_test)

In [11]:
dtype = torch.float64

train_loader = data.DataLoader(
    data.TensorDataset(
        torch.tensor(scaled_train, dtype=dtype), 
        torch.tensor(Y, dtype=dtype)), 
    batch_size = 8,
    shuffle = True)

x_train = train_loader.dataset.tensors[0]
y_train = train_loader.dataset.tensors[1]

x_test = torch.tensor(scaled_test, dtype=dtype)
y_test = torch.tensor(Y_test, dtype=dtype)

In [12]:
x_train.dtype

torch.float64

## Model & Training

### ANFIS

In [13]:
model = nft.rule_reduced_ANFIS(
    input_size = 2,
    num_mfs = 2,
    outputs = 1,
    dtype=torch.float64
)

In [14]:
model.init_premises(x_train)

### Hybrid Learning Algorithm

In [15]:
loss_fn = nn.functional.mse_loss
#loss_fn = nn.functional.binary_cross_entropy
#loss_fn = nn.functional.cross_entropy

optimizer = torch.optim.AdamW
params = {'lr': 0.001, 'weight_decay': 0.01}

early_stopping = nft.EarlyStopping(patience=5, delta=0.001)

In [16]:
trainer = nft.Hybrid_learning_algorithm(
    epochs=20,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    early_stopping=early_stopping
)

### SONFIS

In [17]:
Ngrow = 60
dGrow = 0.7
Nsplit = 50
eSplit = 0.8
Nvanish = 6
lVanish = 2

max_iterations = 40

anfis_trainer = trainer

validation = 0.3
sonfis_early_stopping = nft.EarlyStopping(patience=8)
last_training_iteration = True

In [18]:
sonfis = nft.SONFIS(
    Ngrow=Ngrow,
    dGrow=dGrow,
    Nsplit=Nsplit,
    eSplit=eSplit,
    Nvanish=Nvanish,
    lVanish=lVanish,
    max_iterations=max_iterations,
    ANFIStrainer=anfis_trainer,
    validation=validation,
    early_stopping=sonfis_early_stopping,
    last_training_iteration=last_training_iteration
)

In [19]:
%time sonfis(model, train_loader, verbose=True)

Iteration:  0/40 - loss: 3.179922 - validation loss: 3.548309
 -> ANFIS rules: 2

Iteration:  1/40 - loss: 1.826151 - validation loss: 1.899896
 -> ANFIS rules: 4

Iteration:  2/40 - loss: 1.181729 - validation loss: 1.257398
 -> ANFIS rules: 5

Iteration:  3/40 - loss: 2.035982 - validation loss: 2.183766
 -> ANFIS rules: 6

Iteration:  4/40 - loss: 1.445111 - validation loss: 1.624248
 -> ANFIS rules: 9

Iteration:  5/40 - loss: 0.872848 - validation loss: 0.853109
 -> ANFIS rules: 11

Iteration:  6/40 - loss: 4.714424 - validation loss: 4.830909
 -> ANFIS rules: 9

Iteration:  7/40 - loss: 4.873762 - validation loss: 4.817360
 -> ANFIS rules: 13

Iteration:  8/40 - loss: 2.818817 - validation loss: 2.710293
 -> ANFIS rules: 14

Iteration:  9/40 - loss: 2.565116 - validation loss: 2.395138
 -> ANFIS rules: 15

Iteration: 10/40 - loss: 3.290140 - validation loss: 3.249411
 -> ANFIS rules: 17

Iteration: 11/40 - loss: 2.094064 - validation loss: 1.968677
 -> ANFIS rules: 18

Iteration:

In [20]:
test_measures = nft.get_measures(model, x_test, y_test)

for measure in test_measures:
    print(measure + ':', test_measures[measure])

MSE: 0.10890523373673787
RMSE: 0.3300079298088727
MAE: 0.24064984811785475
R2: 0.96709068562894
MAPE: 37.165448578147526


In [21]:
train_measures = nft.get_measures(model, x_train, y_train)

for measure in train_measures:
    print(measure + ':', train_measures[measure])

MSE: 0.3458879040771195
RMSE: 0.5881223546823565
MAE: 0.464049683215264
R2: 0.9087481925140277
MAPE: 1.4532919797976187
